# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import os
import sys

repo_url  = "https://github.com/Seongha-parkk/2026-HYU-AUE8088-PA2.git"
repo_name = "2026-HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    !git -C /content/{repo_name} pull

%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
import os

drive.mount('/drive')

DRIVE_CKPT = "/drive/MyDrive/aue8088-pa2/checkpoints"
os.makedirs(DRIVE_CKPT, exist_ok=True)

LOCAL_CKPT = os.path.abspath("../checkpoints")
if os.path.islink(LOCAL_CKPT):
    print(f"symlink 이미 존재: {LOCAL_CKPT} → {os.readlink(LOCAL_CKPT)}")
elif os.path.isdir(LOCAL_CKPT):
    import shutil
    for f in os.listdir(LOCAL_CKPT):
        shutil.move(os.path.join(LOCAL_CKPT, f), DRIVE_CKPT)
    shutil.rmtree(LOCAL_CKPT)
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)
    print(f"기존 파일 Drive 이전 후 symlink 생성: {LOCAL_CKPT} → {DRIVE_CKPT}")
else:
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)
    print(f"symlink 생성: {LOCAL_CKPT} → {DRIVE_CKPT}")

In [ ]:
import math
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform, IMAGENET_MEAN, IMAGENET_STD
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES, average_macro_f1
from src.utils.submission import write_submission
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES, NUM_CLASSES
from src.datasets.samplers import multi_attr_balanced_sampler
from src.losses.imbalanced import FocalLoss
from src.augment.mix import cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
STRATEGY_NAME = "ir-rarity+uncertainty"
WANDB_TAGS    = ["level5", STRATEGY_NAME]

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


In [ ]:
# 1단계 — best 모델(level3 ViT)로 Set B 스코어링
BEST_CKPT = "../checkpoints/level3_ir-focal+ir-sampler+randaug+cutmix.pth"
model = vit_small_patch16_224().to(device)
model.load_state_dict(torch.load(BEST_CKPT, map_location=device)["state_dict"])
model.eval()

# Set A 클래스 수 & IR 계산 (rarity score 기준)
train_ds_a = BDDAttrDataset("../data/set_a", "train", transform=eval_transform())
set_a_counts = {a: train_ds_a.class_counts(a).float() for a in ATTRIBUTES}
IR = {a: (set_a_counts[a].max() / set_a_counts[a][set_a_counts[a] > 0].min()).item()
      for a in ATTRIBUTES}
ir_total = sum(IR.values())
print("Set A IR:", {a: round(v, 1) for a, v in IR.items()})

# Set B 전체 로드 & 예측
set_b    = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)
preds_b, probs_b, _, ids_b = collect_predictions(model, loader_b, device)

# ── Signal 1: IR-weighted rarity score (Set A 희귀 클래스 보충 우선) ──────
rarity = np.zeros(len(set_b))
for i, s in enumerate(set_b.samples):
    for a in ATTRIBUTES:
        label = getattr(s, a)
        if label >= 0:
            rarity[i] += (IR[a] / ir_total) / max(set_a_counts[a][label].item(), 1)

# ── Signal 2: Uncertainty (모델이 틀릴 가능성이 높은 샘플 우선) ───────────
max_probs   = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)
uncertainty = 1.0 - max_probs.mean(axis=1)

# ── 결합 점수 (정규화 후 가중 합산) ─────────────────────────────────────
def norm01(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

score = 0.7 * norm01(rarity) + 0.3 * norm01(uncertainty)
print(f"Set B 총 {len(set_b)}장 | rarity mean={rarity.mean():.4f} | unc mean={uncertainty.mean():.3f}")

In [ ]:
# 2단계 — 결합 점수 상위 1,000장 선별
K     = 1000
order = np.argsort(-score)[:K]

picks = []
for i in order:
    s = set_b.samples[i]
    picks.append({
        "image_id":  s.image_id,
        "weather":   int(s.weather),
        "scene":     int(s.scene),
        "timeofday": int(s.timeofday),
        "score":     float(score[i]),
        "rarity":    float(rarity[i]),
        "uncertainty": float(uncertainty[i]),
    })

with open("../level5_picks.json", "w") as f:
    json.dump({
        "strategy": (
            "IR-weighted rarity score (0.7) + model uncertainty (1-max_softmax, 0.3) 결합. "
            "Set A의 속성별 IR을 기준으로 희귀 클래스 조합을 가진 이미지에 높은 가중치를 부여하고, "
            "level3 ViT 모델이 예측에 불확실한 샘플을 추가로 우선 선택."
        ),
        "num_picks": len(picks),
        "picks": picks,
    }, f, indent=2)

# 선별 결과 분포 확인
from collections import Counter
print(f"\n선별된 {K}장 분포:")
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    dist = {CLASS_NAMES[a][k]: cnt.get(k, 0) for k in range(NUM_CLASSES[a])}
    print(f"  {a}: {dist}")

---
## 전략 선택 셀 (B~F)

아래 셀 중 **하나만** 실행해 `picks` / `STRATEGY_NAME` 을 덮어쓴 뒤 **retrain 셀**을 실행하세요.  
여러 전략을 비교하려면 순서대로 실행하면 wandb 에 Run 이 누적됩니다.

| 셀 | 전략 | 설명 |
|---|---|---|
| **A** (위) | `ir-rarity+uncertainty` | IR 가중 희귀도 0.7 + 불확실성 0.3 |
| **B** | `quota-deficit` | 속성별 부족량 비례 예산 배분 |
| **C** | `triplet-rarity` | 3속성 조합 빈도 기반 희귀도 |
| **D** | `high-loss` | true-label CE loss 최대 샘플 |
| **E** | `minority-error` | 소수 클래스 오예측 집중 |
| **F** | `random` | 무작위 (DI baseline) |

In [ ]:
# ── 전략 B: Quota-Deficit ────────────────────────────────────────────────────
# 속성별로 Set A 최다 클래스 기준 부족량(deficit)을 계산하고,
# 1000 예산을 deficit 비율에 맞춰 배분. 각 슬롯 내에서는 uncertainty 높은 순 선택.

STRATEGY_NAME = "quota-deficit"
WANDB_TAGS    = ["level5", STRATEGY_NAME]

K = 1000

# 속성별 deficit 계산
deficits = {}
for a in ATTRIBUTES:
    max_c = int(set_a_counts[a].max().item())
    deficits[a] = {c: max(0, max_c - int(set_a_counts[a][c].item()))
                   for c in range(NUM_CLASSES[a])}
total_deficit = sum(sum(d.values()) for d in deficits.values())

# 각 (속성, 클래스) 슬롯에 배정할 예산
slots = {}
for a in ATTRIBUTES:
    for c, def_c in deficits[a].items():
        slots[(a, c)] = round(K * def_c / max(total_deficit, 1))

# Set B 인덱스를 (속성, 클래스)별로 분류 후 uncertainty 내림차순 정렬
from collections import defaultdict
bucket = defaultdict(list)  # (a, c) → [(unc, idx)]
for i, s in enumerate(set_b.samples):
    for a in ATTRIBUTES:
        label = getattr(s, a)
        if label >= 0:
            bucket[(a, label)].append((uncertainty[i], i))
for key in bucket:
    bucket[key].sort(reverse=True)

# 슬롯에 따라 그리디하게 선택 (중복 방지)
selected = set()
for (a, c), quota in sorted(slots.items(), key=lambda x: -x[1]):
    for unc, idx in bucket[(a, c)]:
        if idx not in selected:
            selected.add(idx)
        if len(selected) >= K:
            break
    if len(selected) >= K:
        break

# 부족하면 uncertainty 순으로 보충
if len(selected) < K:
    for idx in np.argsort(-uncertainty):
        if idx not in selected:
            selected.add(idx)
        if len(selected) >= K:
            break

picks = []
for i in list(selected)[:K]:
    s = set_b.samples[i]
    picks.append({"image_id": s.image_id, "weather": int(s.weather),
                  "scene": int(s.scene), "timeofday": int(s.timeofday)})

with open("../level5_picks.json", "w") as f:
    json.dump({"strategy": STRATEGY_NAME, "num_picks": len(picks), "picks": picks}, f, indent=2)

from collections import Counter
print(f"[{STRATEGY_NAME}] {len(picks)}장 선별")
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    print(f"  {a}: { {CLASS_NAMES[a][k]: cnt.get(k,0) for k in range(NUM_CLASSES[a])} }")

In [ ]:
# ── 전략 C: Triplet Rarity ───────────────────────────────────────────────────
# (weather, scene, timeofday) 3속성 조합의 Set A 빈도 역수로 채점.
# 속성별 IR보다 세밀하게 "실제로 드문 조합"을 식별.

STRATEGY_NAME = "triplet-rarity"
WANDB_TAGS    = ["level5", STRATEGY_NAME]

K = 1000

# Set A 조합 빈도 카운트
from collections import Counter as _Counter
triplet_counts = _Counter()
for s in train_ds_a.samples:
    triplet_counts[(s.weather, s.scene, s.timeofday)] += 1

# Set B 각 이미지에 triplet rarity score 부여
triplet_score = np.zeros(len(set_b))
for i, s in enumerate(set_b.samples):
    key = (s.weather, s.scene, s.timeofday)
    triplet_score[i] = 1.0 / (triplet_counts.get(key, 0) + 1)

# uncertainty와 결합 (triplet rarity 0.7 + uncertainty 0.3)
score_c = 0.7 * norm01(triplet_score) + 0.3 * norm01(uncertainty)
order_c = np.argsort(-score_c)[:K]

picks = []
for i in order_c:
    s = set_b.samples[i]
    picks.append({"image_id": s.image_id, "weather": int(s.weather),
                  "scene": int(s.scene), "timeofday": int(s.timeofday),
                  "triplet_count_in_setA": int(triplet_counts.get((s.weather, s.scene, s.timeofday), 0))})

with open("../level5_picks.json", "w") as f:
    json.dump({"strategy": STRATEGY_NAME, "num_picks": len(picks), "picks": picks}, f, indent=2)

print(f"[{STRATEGY_NAME}] {len(picks)}장 선별")
print(f"  선별된 조합 중 Set A에 0개인 triplet: {sum(1 for p in picks if p['triplet_count_in_setA']==0)}장")
from collections import Counter
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    print(f"  {a}: { {CLASS_NAMES[a][k]: cnt.get(k,0) for k in range(NUM_CLASSES[a])} }")

In [ ]:
# ── 전략 D: High-Loss (true-label CE loss) ───────────────────────────────────
# 실제 라벨 기준으로 CE loss를 계산해 가장 헷갈리는 샘플을 선택.
# uncertainty(max-softmax)보다 정확한 "모델이 얼마나 틀렸는지" 측정.
# IR로 정규화해 다수 클래스가 점수를 독식하지 않도록 조정.

STRATEGY_NAME = "high-loss"
WANDB_TAGS    = ["level5", STRATEGY_NAME]

import torch.nn.functional as F

K = 1000
ce_loss = np.zeros(len(set_b))

model.eval()
loader_b_idx = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)
offset = 0
with torch.no_grad():
    for batch in loader_b_idx:
        imgs = batch["image"].to(device)
        logits = model(imgs)
        B = imgs.size(0)
        loss_sum = torch.zeros(B, device=device)
        for a in ATTRIBUTES:
            labels = batch[a].to(device)
            loss_a = F.cross_entropy(logits[a], labels, reduction="none")
            # IR 정규화: 희귀 속성의 loss를 상대적으로 증폭
            loss_sum += (IR[a] / ir_total) * loss_a
        ce_loss[offset: offset + B] = loss_sum.cpu().numpy()
        offset += B

order_d = np.argsort(-ce_loss)[:K]

picks = []
for i in order_d:
    s = set_b.samples[i]
    picks.append({"image_id": s.image_id, "weather": int(s.weather),
                  "scene": int(s.scene), "timeofday": int(s.timeofday),
                  "ce_loss": float(ce_loss[i])})

with open("../level5_picks.json", "w") as f:
    json.dump({"strategy": STRATEGY_NAME, "num_picks": len(picks), "picks": picks}, f, indent=2)

print(f"[{STRATEGY_NAME}] {len(picks)}장 선별 | mean CE loss={ce_loss[order_d].mean():.4f}")
from collections import Counter
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    print(f"  {a}: { {CLASS_NAMES[a][k]: cnt.get(k,0) for k in range(NUM_CLASSES[a])} }")

In [ ]:
# ── 전략 E: Minority-Error Focus ─────────────────────────────────────────────
# 소수 클래스 이미지 중 모델이 실제로 오예측한 샘플을 1순위로 선택.
# 모델의 약점 + 데이터 희귀성을 동시에 공략.
# 순위: ① 소수 클래스 오예측 → ② 소수 클래스 고불확실 → ③ 전체 고불확실

STRATEGY_NAME = "minority-error"
WANDB_TAGS    = ["level5", STRATEGY_NAME]

K = 1000

# 소수 클래스 정의: 각 속성에서 Set A 중앙값 미만인 클래스
minority_labels = {}
for a in ATTRIBUTES:
    median_c = set_a_counts[a][set_a_counts[a] > 0].median().item()
    minority_labels[a] = {c for c in range(NUM_CLASSES[a])
                          if set_a_counts[a][c].item() < median_c}
print("소수 클래스:", {a: [CLASS_NAMES[a][c] for c in minority_labels[a]] for a in ATTRIBUTES})

# 각 이미지의 tier 결정
tier = np.full(len(set_b), 3)  # default: tier 3 (전체 고불확실)
for i, s in enumerate(set_b.samples):
    is_minority = any(getattr(s, a) in minority_labels[a] for a in ATTRIBUTES)
    is_wrong    = any(preds_b[a][i] != getattr(s, a) for a in ATTRIBUTES)
    if is_minority and is_wrong:
        tier[i] = 1   # 소수 클래스 + 오예측
    elif is_minority:
        tier[i] = 2   # 소수 클래스 (정예측이라도 추가 가치 있음)

# tier 오름차순 → 동일 tier 내에서는 uncertainty 내림차순
sort_key = tier * 10 - uncertainty  # tier=1이 가장 작아 먼저 선택됨
order_e = np.argsort(sort_key)[:K]

picks = []
for i in order_e:
    s = set_b.samples[i]
    picks.append({"image_id": s.image_id, "weather": int(s.weather),
                  "scene": int(s.scene), "timeofday": int(s.timeofday),
                  "tier": int(tier[i])})

with open("../level5_picks.json", "w") as f:
    json.dump({"strategy": STRATEGY_NAME, "num_picks": len(picks), "picks": picks}, f, indent=2)

from collections import Counter
tier_cnt = Counter(p["tier"] for p in picks)
print(f"[{STRATEGY_NAME}] tier1(소수+오류)={tier_cnt[1]}  tier2(소수)={tier_cnt[2]}  tier3(기타)={tier_cnt[3]}")
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    print(f"  {a}: { {CLASS_NAMES[a][k]: cnt.get(k,0) for k in range(NUM_CLASSES[a])} }")

In [ ]:
# ── 전략 F: Random (DI score 분모 baseline) ──────────────────────────────────
# 무작위 선택. 다른 전략들의 DI score 계산을 위한 baseline run.

STRATEGY_NAME = "random"
WANDB_TAGS    = ["level5", STRATEGY_NAME]

K = 1000
rng = np.random.default_rng(SEED)
random_idx = rng.choice(len(set_b), K, replace=False)

picks = []
for i in random_idx:
    s = set_b.samples[i]
    picks.append({"image_id": s.image_id, "weather": int(s.weather),
                  "scene": int(s.scene), "timeofday": int(s.timeofday)})

with open("../level5_picks.json", "w") as f:
    json.dump({"strategy": STRATEGY_NAME, "num_picks": len(picks), "picks": picks}, f, indent=2)

from collections import Counter
print(f"[{STRATEGY_NAME}] {len(picks)}장 랜덤 선별")
for a in ATTRIBUTES:
    cnt = Counter(p[a] for p in picks)
    print(f"  {a}: { {CLASS_NAMES[a][k]: cnt.get(k,0) for k in range(NUM_CLASSES[a])} }")

In [ ]:
# 3단계 — Set A + picks 로 재학습 (level3 전략 동일 적용)
extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]

def train_transform_l5(img_size=224):
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

train_aug = BDDAttrDataset("../data/set_a", "train", transform=train_transform_l5(), extra_picks=extra)
val_ds    = BDDAttrDataset("../data/set_a", "val",   transform=eval_transform())

# 결합 데이터셋 기준으로 IR 재계산
IR2 = {a: (train_aug.class_counts(a).float().max()
           / train_aug.class_counts(a).float().clamp(min=1e-6).min()).item()
       for a in ATTRIBUTES}
gamma_dict  = {a: round(1.0 + math.log(IR2[a]), 1) for a in ATTRIBUTES}
ir2_total   = sum(IR2.values())
mix_weights = {a: IR2[a] / ir2_total for a in ATTRIBUTES}
print("결합 데이터셋 IR:", {a: round(v, 1) for a, v in IR2.items()})
print("FocalLoss gamma:", gamma_dict)

loss_fns = {a: FocalLoss(gamma=gamma_dict[a]).to(device) for a in ATTRIBUTES}
sampler2 = multi_attr_balanced_sampler(train_aug, IR2)

g = torch.Generator(); g.manual_seed(SEED)
loader_tr  = DataLoader(train_aug, batch_size=64, sampler=sampler2, num_workers=2, pin_memory=True)
loader_val = DataLoader(val_ds,    batch_size=64, shuffle=False,    num_workers=2, pin_memory=True)

epochs = 30
set_seed(SEED, deterministic=True)
model2 = vit_small_patch16_224().to(device)
optim  = torch.optim.AdamW(model2.parameters(), lr=5e-4, weight_decay=5e-2)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level5-{STRATEGY_NAME}",
    config={
        "backbone": "vit_s16", "strategy": STRATEGY_NAME,
        "num_picks": len(picks), "epochs": epochs, "batch": 64,
        "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
        "IR_combined": {a: round(v, 1) for a, v in IR2.items()},
        "gamma": gamma_dict,
    },
    tags=WANDB_TAGS,
)
trainer = MultiTaskTrainer(model2, optim, sched, loss_fns, device, TrainConfig(epochs=epochs), logger=logger)

# CutMix 학습 루프
def step_with_mix(images, targets):
    x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model2(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam, weights=mix_weights)

history = {"train_loss": [], "val_avg_mf1": []}
for epoch in range(epochs):
    model2.train()
    running, n_batches = 0.0, 0
    pbar = tqdm(loader_tr, desc=f"train e{epoch+1:02d}", leave=False)
    for batch in pbar:
        images  = batch["image"].to(device, non_blocking=True)
        targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}
        optim.zero_grad(set_to_none=True)
        loss = step_with_mix(images, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        optim.step()
        running += loss.item(); n_batches += 1
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss  = running / max(n_batches, 1)
    val_metrics = trainer.evaluate(loader_val)
    sched.step()
    history["train_loss"].append(train_loss)
    history["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
    print(f"[epoch {epoch+1:02d}/{epochs}] loss={train_loss:.4f}  val_avg_MF1={val_metrics['avg_macro_f1']:.4f}")
    log_payload = {"epoch": epoch+1, "train/loss": train_loss,
                   "val/avg_macro_f1": val_metrics["avg_macro_f1"],
                   "lr": optim.param_groups[0]["lr"]}
    for a, v in val_metrics["per_macro_f1"].items():
        log_payload[f"val/mf1_{a}"] = v
    logger.log(log_payload, step=epoch)

# 학습 종료 후 wandb 업로드
val_pred, _, val_tgt, _ = collect_predictions(model2, loader_val, device)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])

from collections import Counter
for a in ATTRIBUTES:
    cnt  = Counter(p[a] for p in picks)
    rows = [[CLASS_NAMES[a][k], cnt.get(k, 0)] for k in range(NUM_CLASSES[a])]
    logger.log_table(f"picks/distribution_{a}", ["class", "count"], rows)

os.makedirs("../checkpoints", exist_ok=True)
torch.save({"state_dict": model2.state_dict(), "history": history},
           "../checkpoints/level5_final.pth")
logger.finish()
print("저장 완료 → ../checkpoints/level5_final.pth")

In [ ]:
# 4단계 — Kaggle 제출용 CSV 생성.
test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.